In [43]:
import pandas as pd
import numpy as np
import sys
import json
sys.path.insert(0, "../../utils/")
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV
from joblib import dump
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    roc_auc_score
)
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb
import lightgbm as lgb

param_grid = {
    "n_estimators": [100, 500, 1000], 
    "criterion": ["gini", "entropy"],           
    "min_samples_split": [2, 5, 10],           
    "min_samples_leaf": [1, 2, 4],              
    "max_features": ["sqrt", "log2"],  
    "max_depth": [10, 20, 30]
    }

In [44]:
MODELS = {
    "RandomForest": RandomForestClassifier,
    "AdaBoost": AdaBoostClassifier,
    "SVM": SVC,
    "GradientBoosting": GradientBoostingClassifier,
    "LogisticRegression": LogisticRegression,
    "XGBoost": xgb.XGBClassifier,
    "LGBM": lgb.LGBMClassifier,
    "KNN": KNeighborsClassifier
}

In [ ]:
GRIDS = {
    "RandomForest": {
        "n_estimators": [100, 500, 1000, 3000, 5000],
        "criterion": ["gini", "entropy"],
        "min_samples_split": [2, 10, 20],
        "min_samples_leaf": [1, 4, 8],
        "max_features": ["sqrt", "log2"],
        "max_depth": [10, 30, 50, None]
    },
    "AdaBoost": {
        "n_estimators": [50, 200, 500, 1000],
        "learning_rate": [0.01, 0.1, 1.0]
    },
    "GradientBoosting": {
        "n_estimators": [100, 300, 500],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 5],
        "subsample": [0.8, 1.0]
    },
    "XGBoost": {
        "n_estimators": [100, 300, 500],
        "max_depth": [3, 5],
        "learning_rate": [0.01, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0]
    },
    "LGBM": {
        "n_estimators": [100, 300, 500],
        "max_depth": [3, 5, -1],
        "learning_rate": [0.01, 0.1],
        "num_leaves": [15, 31],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0]
    },
    "KNN": {
        "n_neighbors": [3, 5, 7],
        "weights": ["uniform", "distance"],
        "metric": ["euclidean", "manhattan"]
    },
    "SVM": {
        "C": [0.1, 1, 10],
        "kernel": ["linear", "rbf"],
        "gamma": ["scale"]
    },
    "LogisticRegression": {
        "C": [0.1, 1, 10],
        "solver": ["liblinear"],
        "penalty": ["l1", "l2"]
    }
}

In [46]:
param_grid = {
    "n_estimators": [100]
    }

In [47]:
def undersampling(df_data, seed):
    #ten_percent=df_data[df_data["target"]==2].sample(frac=0.10, random_state=seed)
    X=df_data.drop('target', axis=1)
    y=df_data['target']  
    #Se definen los objetos para submuestrear
    X['index']=X.index 
    undersampler=RandomUnderSampler(sampling_strategy='not minority', random_state=seed)    

    #Se aplica el submuestreo
    X_res, y_res=undersampler.fit_resample(X, y)
    df_resampled=pd.concat([X_res,y_res], axis=1)

    index_res=X_res['index']
    df_resampled.drop('index', axis=1, inplace=True)

    mask=~X['index'].isin(index_res)
    excluded_data=df_data[mask.values]
    #data_independent= pd.concat([ten_percent, excluded_data], axis=0)
    data_independent= pd.concat([excluded_data], axis=0)
    data_independent.reset_index(drop=True, inplace=True)
    data_independent.to_csv("../../models/data/data_independent.csv", index=False)
    
    return df_resampled

In [48]:
def oversampling(df_data, seed):
    X = df_data.drop('target', axis=1)
    y = df_data['target']  
    #Se definen los objetos para sobremuestrear
    smote = SMOTE(random_state=seed)

    #Se aplica el sobremuestreo
    X_res, y_res= smote.fit_resample(X, y)
    df_resampled = pd.concat([X_res,y_res], axis=1)
    
    return df_resampled

In [ ]:
def function_split(df_data, seed):
    #Separa los datos
    data_under= undersampling(df_data, seed)
    data_over= oversampling(df_data, seed)
    train_data, val_data = train_test_split(df_data, test_size=0.2, random_state=seed)
    train_data_under, val_data_under = train_test_split(data_under, test_size=0.2, random_state=seed)
    train_data_over, val_data_over = train_test_split(data_over, test_size=0.2, random_state=seed)
    return train_data, val_data, train_data_under, val_data_under, train_data_over, val_data_over

In [50]:
def metrics(model, predict_val, y_val, dataset, div, predict_proba=None):
    acc_value = accuracy_score(y_pred=predict_val, y_true=y_val) 
    recall_value = recall_score(y_pred=predict_val, y_true=y_val, average='weighted')
    precision_value = precision_score(y_pred=predict_val, y_true=y_val, average='weighted') 
    f1_value = f1_score(y_pred=predict_val, y_true=y_val, average='weighted')
    mcc_value = matthews_corrcoef(y_pred=predict_val, y_true=y_val)
    cm = confusion_matrix(y_pred=predict_val, y_true=y_val)
    cm_dict = pd.DataFrame(cm).to_dict()

    roc_auc_value = None
    if predict_proba is not None:
        try:
            roc_auc_value = roc_auc_score(y_val, predict_proba, multi_class='ovr', average='weighted')
        except Exception as e:
            print(f"Error computing ROC AUC: {e}")
            roc_auc_value = None

    df_metrics = pd.DataFrame([[dataset, model, div, acc_value, recall_value, precision_value, f1_value, mcc_value, roc_auc_value, cm_dict]],
                              columns=["dataset", "model", "sampling", "acc", "recall", "precision", "f1", "mcc", "roc_auc", "conf_matrix"])

    return df_metrics

In [51]:
def cross(model, X_train, y_train, k):
    scoring_metrics = {
        "accuracy": "accuracy",
        "recall": "recall_weighted",
        "precision": "precision_weighted",
        "f1": "f1_weighted"
    }

    results = {}
    for name, scoring in scoring_metrics.items():
        scores = cross_val_score(model, X_train, y_train, cv=k, scoring=scoring)
        print(f"{name} scores: {scores}")
        results[name] = np.mean(scores)
    results_df = pd.DataFrame([results])
    return results_df

In [ ]:
def function_train(model_name, train, val, div, seed):
    model_cls = MODELS[model_name]

    if model_name == "KNN":
        model = model_cls(n_jobs=-1)
    elif model_name == "SVM":
        model = model_cls(probability=True, random_state=seed)
    elif model_name in ["RandomForest", "XGBoost", "LGBM"]:
        model = model_cls(random_state=seed, n_jobs=-1)
    else:
        model = model_cls(random_state=seed)

    # Separa los datos de entrenamiento y validación
    X_train = train.drop(columns="target").values
    y_train = train["target"].values
    X_val= val.drop(columns="target").values
    y_val = val["target"].values

    # Combina los conjuntos de entrenamiento y validación para la validación cruzada
    train_all= pd.concat([train, val], ignore_index=True)
    target = train_all["target"]
    train_all.drop(columns="target", inplace=True)
    df_combined = val.copy()

    print(f"Train {model_name} with seed {seed} and division {div}")    
    results = []

    #Valdación cruzada antes de la búsqueda de hiperparámetros
    model.fit(train_all, target)
    cv_scores = cross(model, train_all, target, k=10)
    cv_scores["set"] = "CrossVal"
    cv_scores["model"] = model_name
    cv_scores["sampling"] = div
    results.append(cv_scores)
    
    #cv_scores.to_csv(f"../../models/data/metrics/{model_name}_{seed}_{div}_cross_val.csv", index=False)

    dump(model, f"../../models/{model_name}_{seed}_{div}.joblib")

    #Se predice en el conjunto de entrenamiento y validación
    y_pred_train = model.predict(X_train)
    y_proba_train = model.predict_proba(X_train)

    y_pred_val = model.predict(X_val)
    y_proba_val = model.predict_proba(X_val)

    #Se obtiene un conjunto combinado de datos con las predicciones
    df_combined.drop(columns="target", inplace=True)
    df_combined["y_true"] = y_val
    df_combined["y_pred"] = y_pred_val
    df_combined.to_csv(f"../../models/data/{model_name}_{seed}_{div}_predictions.csv", index=False)

    #Se guardan las métricas de entrenamiento y validación
    train_metrics = metrics(model_name, y_pred_train, y_train, "Train", div, predict_proba=y_proba_train)
    val_metrics = metrics(model_name, y_pred_val, y_val, "Validation", div, predict_proba=y_proba_val)
    results.append(train_metrics)
    results.append(val_metrics)

    all_results = pd.concat(results, ignore_index=True)
    all_results.to_csv(f"../../models/data/metrics/{model_name}_{seed}_{div}_all_metrics.csv", index=False)

    return all_results, model

In [ ]:
def grid_function(model_name, model, train, val, seed, div):
    grid = GridSearchCV(estimator=model, param_grid=GRIDS[model_name], cv=5, scoring="f1_weighted", n_jobs=-1)
    X_train = train.drop(columns="target").values
    y_train = train["target"].values
    X_val= val.drop(columns="target").values
    y_val = val["target"].values

    df_combined = val.copy()
    print(f"GridSearchCV for Random Forest with seed {seed} and division {div}")
    # Se realiza la búsqueda de hiperparámetros
    grid.fit(X_train, y_train)

    # Se obtienen los mejores parámetros y el mejor modelo
    best_model = grid.best_estimator_
    best_params = grid.best_params_
    best_score = grid.best_score_

    best_params = json.dumps(best_params, indent=4)
    with open(f"../../models/data/json/{model_name}_Grid_{seed}_{div}_best_params.json", "w") as f:
        f.write(best_params)

    print(f"Best estimators: {best_model} and best parameters found: {best_params}")

    dump(best_model, f"../../models/data/best/{model_name}_Grid_{seed}_{div}_best.joblib")

    # Se predice en el conjunto de entrenamiento y validación
    y_pred_train = best_model.predict(X_train)
    y_proba_train = best_model.predict_proba(X_train)

    y_pred_val = best_model.predict(X_val)
    y_proba_val = best_model.predict_proba(X_val)

    # Se obtiene un conjunto combinado de datos con las predicciones
    df_combined.drop(columns="target", inplace=True)
    df_combined["y_true"] = y_val
    df_combined["y_pred"] = y_pred_val
    df_combined.to_csv(f"../../models/data/{model_name}_Grid_{seed}_{div}_best_predictions.csv", index=False)

    # Se guardan las métricas de entrenamiento y validación
    train_metrics = metrics(model_name, y_pred_train, y_train, "Train", div, predict_proba=y_proba_train)
    val_metrics = metrics(model_name, y_pred_val, y_val, "Validation", div, predict_proba=y_proba_val)
    results = pd.concat([train_metrics, val_metrics], ignore_index=True)
    return results

In [ ]:
def main_train(df_data, seed, model_name, grid_search=False):
    all_metrics = []
    all_metrics_grid = []
    df_train, df_val, df_train_under, df_val_under, df_train_over, df_val_over = function_split(df_data, seed)
    metrics_orig, model_orig = function_train(model_name, df_train, df_val, "Original", seed)
    metrics_under, model_under = function_train(model_name, df_train_under, df_val_under, "Under", seed)
    metrics_over, model_over = function_train(model_name, df_train_over, df_val_over, "Over", seed)

    all_metrics = pd.concat([metrics_orig,metrics_under,metrics_over], ignore_index=True)
    all_metrics.to_csv(f"../../metrics/{model_name}_Grid_{seed}_metrics.csv", index=False)
    if grid_search:
        all_metrics_grid = pd.concat([grid_function(model_name, model_orig, df_train, df_val, seed, "Original"),
                                      grid_function(model_name, model_under, df_train_under, df_val_under, seed, "Under"),
                                      grid_function(model_name, model_over, df_train_over, df_val_over, seed, "Over")], 
                                      ignore_index=True)
        all_metrics_grid.to_csv(f"../../metrics/{model_name}_{seed}_grid_metrics.csv", index=False)    

In [55]:
repr_name="embedding_antiviral_homology_90_protT5"
df_data = pd.read_csv(f"../../data/numerical_rep/{repr_name}.csv")
df_data.drop(["experimental_characteristics"], axis=1, inplace=True)
seed= 42

In [56]:
for model_name in MODELS.keys():
    print(f"Processing {model_name} with {repr_name}")
    main_train(df_data, seed, model_name, grid_search=True)
    print(f"Finished {model_name}")
    print("=====================================")

Processing RandomForest with embedding_antiviral_homology_90_protT5
Train Random Forest with seed 42 and division Original
accuracy scores: [0.37121212 0.31060606 0.38636364 0.39694656 0.38931298 0.28244275
 0.29770992 0.41221374 0.3740458  0.36641221]
recall scores: [0.37121212 0.31060606 0.38636364 0.39694656 0.38931298 0.28244275
 0.29770992 0.41221374 0.3740458  0.36641221]
precision scores: [0.33004149 0.2870725  0.34794372 0.35220724 0.31843783 0.268125
 0.28777073 0.3652747  0.3221349  0.33915691]
f1 scores: [0.34631402 0.29755978 0.36220611 0.36874179 0.34794574 0.27297793
 0.2924952  0.38156224 0.34194124 0.34990068]
Train Random Forest with seed 42 and division Under


/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


accuracy scores: [0.33333333 0.35       0.33333333 0.36666667 0.33898305 0.30508475
 0.47457627 0.25423729 0.33898305 0.3220339 ]
recall scores: [0.33333333 0.35       0.33333333 0.36666667 0.33898305 0.30508475
 0.47457627 0.25423729 0.33898305 0.3220339 ]
precision scores: [0.33862434 0.3927847  0.31764069 0.36688596 0.33918775 0.2836034
 0.4595015  0.25342155 0.32149866 0.30372102]
f1 scores: [0.33547283 0.36179877 0.32217129 0.36324786 0.33805791 0.28854445
 0.46145838 0.25329567 0.32617665 0.3050237 ]
Train Random Forest with seed 42 and division Over


/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


accuracy scores: [0.54901961 0.54411765 0.53921569 0.5        0.48768473 0.52216749
 0.5270936  0.54187192 0.57142857 0.56650246]
recall scores: [0.54901961 0.54411765 0.53921569 0.5        0.48768473 0.52216749
 0.5270936  0.54187192 0.57142857 0.56650246]
precision scores: [0.53725459 0.53334634 0.53319004 0.49023171 0.48638074 0.53403138
 0.49719962 0.52974153 0.55812704 0.55770783]
f1 scores: [0.54226021 0.53654009 0.53546458 0.49454605 0.48639679 0.52716413
 0.50206452 0.53413166 0.56199247 0.56086565]


...

/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


accuracy scores: [0.56372549 0.55882353 0.59313725 0.51960784 0.57635468 0.57142857
 0.56157635 0.61083744 0.59605911 0.62068966]
recall scores: [0.56372549 0.55882353 0.59313725 0.51960784 0.57635468 0.57142857
 0.56157635 0.61083744 0.59605911 0.62068966]
precision scores: [0.54158106 0.54431244 0.58333333 0.49707792 0.56739101 0.56326723
 0.54651599 0.59884242 0.58135898 0.60939182]
f1 scores: [0.54378741 0.54959755 0.58574341 0.50239954 0.57052265 0.56567866
 0.53756366 0.60052997 0.58349102 0.61243244]


/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


GridSearchCV for Random Forest with seed 42 and division Original
Best estimators: LogisticRegression(C=0.1, random_state=42, solver='liblinear') and best parameters found: {
    "C": 0.1,
    "solver": "liblinear"
}
GridSearchCV for Random Forest with seed 42 and division Under
Best estimators: LogisticRegression(C=0.1, random_state=42, solver='liblinear') and best parameters found: {
    "C": 0.1,
    "solver": "liblinear"
}
GridSearchCV for Random Forest with seed 42 and division Over
Best estimators: LogisticRegression(C=1, random_state=42, solver='liblinear') and best parameters found: {
    "C": 1,
    "solver": "liblinear"
}
Finished LogisticRegression
Processing XGBoost with embedding_antiviral_homology_90_protT5
Train Random Forest with seed 42 and division Original
accuracy scores: [0.38636364 0.36363636 0.40151515 0.40458015 0.41221374 0.33587786
 0.36641221 0.40458015 0.44274809 0.41221374]
recall scores: [0.38636364 0.36363636 0.40151515 0.40458015 0.41221374 0.33587786
 0

TypeError: __init__() got an unexpected keyword argument 'random_state'